In [26]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [27]:
load_dotenv()

True

In [28]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [29]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [30]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [31]:

def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [32]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [33]:

config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with *dough* experience and a *crust* of confidence! 🍕😄',
 'explanation': '**Explanation of the joke**\n\n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone with **dough** experience and a **crust** of confidence! 🍕😄*\n\n---\n\n### 1. The set‑up  \nThe joke starts with a classic “Why did X do Y?” format, which primes the listener to expect a funny or unexpected reason for the action. Here, the “X” is a pizza—an inanimate food item—so the idea of it applying for a job is already absurd and sets the stage for wordplay.\n\n### 2. The punchline’s wordplay  \n\n| Phrase in the punchline | Literal meaning (pizza‑related) | Figurative meaning (job‑related) |\n|--------------------------|----------------------------------|-----------------------------------|\n| **dough** experience | Pizza is made from *dough* (the raw, uncooked 

In [34]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with *dough* experience and a *crust* of confidence! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone with **dough** experience and a **crust** of confidence! 🍕😄*\n\n---\n\n### 1. The set‑up  \nThe joke starts with a classic “Why did X do Y?” format, which primes the listener to expect a funny or unexpected reason for the action. Here, the “X” is a pizza—an inanimate food item—so the idea of it applying for a job is already absurd and sets the stage for wordplay.\n\n### 2. The punchline’s wordplay  \n\n| Phrase in the punchline | Literal meaning (pizza‑related) | Figurative meaning (job‑related) |\n|--------------------------|----------------------------------|-----------------------------------|\n| **dough** experience | Pizza is made from *dough* 

In [35]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with *dough* experience and a *crust* of confidence! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone with **dough** experience and a **crust** of confidence! 🍕😄*\n\n---\n\n### 1. The set‑up  \nThe joke starts with a classic “Why did X do Y?” format, which primes the listener to expect a funny or unexpected reason for the action. Here, the “X” is a pizza—an inanimate food item—so the idea of it applying for a job is already absurd and sets the stage for wordplay.\n\n### 2. The punchline’s wordplay  \n\n| Phrase in the punchline | Literal meaning (pizza‑related) | Figurative meaning (job‑related) |\n|--------------------------|----------------------------------|-----------------------------------|\n| **dough** experience | Pizza is made from *dough*

In [36]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti get a promotion?  \n\nBecause it knew how to *pasta* the competition and always *rolled* with the punches! 🍝😄',
 'explanation': '**Explanation of the joke**\n\n| Part of the joke | What it literally says | The wordplay / pun | Why it’s funny |\n|------------------|------------------------|--------------------|----------------|\n| **“Why did the spaghetti get a promotion?”** | Sets up a typical “why‑did‑the‑X‑do‑Y?” format, leading us to expect a workplace‑related answer. | The joke treats spaghetti—a type of pasta—as if it were an employee. | The absurdity of a noodle climbing the corporate ladder is already a bit silly. |\n| **“Because it knew how to *pasta* the competition…”** | “Pasta” is the Italian staple food that includes spaghetti. | The verb “pasta” sounds exactly like “pass” (as in “pass the competition”). The joke swaps the real verb for the food word, creating a homophonic pun. | The listener hears “pass the competition,” a

In [37]:
list(workflow.get_state_history(config2))


[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti get a promotion?  \n\nBecause it knew how to *pasta* the competition and always *rolled* with the punches! 🍝😄', 'explanation': '**Explanation of the joke**\n\n| Part of the joke | What it literally says | The wordplay / pun | Why it’s funny |\n|------------------|------------------------|--------------------|----------------|\n| **“Why did the spaghetti get a promotion?”** | Sets up a typical “why‑did‑the‑X‑do‑Y?” format, leading us to expect a workplace‑related answer. | The joke treats spaghetti—a type of pasta—as if it were an employee. | The absurdity of a noodle climbing the corporate ladder is already a bit silly. |\n| **“Because it knew how to *pasta* the competition…”** | “Pasta” is the Italian staple food that includes spaghetti. | The verb “pasta” sounds exactly like “pass” (as in “pass the competition”). The joke swaps the real verb for the food word, creating a homophonic pun. | The listener hears “pass

Time Travel

In [38]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [39]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with *dough* experience and a *crust* of confidence! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone with **dough** experience and a **crust** of confidence! 🍕😄*\n\n---\n\n### 1. The set‑up  \nThe joke starts with a classic “Why did X do Y?” format, which primes the listener to expect a funny or unexpected reason for the action. Here, the “X” is a pizza—an inanimate food item—so the idea of it applying for a job is already absurd and sets the stage for wordplay.\n\n### 2. The punchline’s wordplay  \n\n| Phrase in the punchline | Literal meaning (pizza‑related) | Figurative meaning (job‑related) |\n|--------------------------|----------------------------------|-----------------------------------|\n| **dough** experience | Pizza is made from *dough*


Updating State

In [40]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1e8c02-58f1-6d5b-8000-3d57d18df473'}}

In [41]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e8c02-58f1-6d5b-8000-3d57d18df473'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-11-25T11:05:38.669696+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='c364eb38-a18f-3a43-7d4c-96a214c0fb9a', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with *dough* experience and a *crust* of confidence! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone with **dough** experience and a **cr

In [42]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e8c02-58f1-6d5b-8000-3d57d18df473'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-11-25T11:05:38.669696+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='c364eb38-a18f-3a43-7d4c-96a214c0fb9a', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with *dough* experience and a *crust* of confidence! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone with **dough** experience and a **cr

Fault tolerance

In [43]:

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [44]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [47]:

# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}